In [1]:
# ============================================================
# 14B_AURORA_ROMA_true_weight_sensitivity.ipynb
#
# True weight-based transaction-cost and rebalance-frequency
# sensitivity analysis for AURORA-TWETF / ROMA-AURORA paper.
#
# Purpose:
# 1. Load final source-aware Notebook 13B strict-test return matrix.
# 2. Load ROMA R2 weights and deduplicate them correctly.
# 3. Auto-discover AURORA Notebook 10 weights where available.
# 4. Re-simulate daily portfolio returns using actual ETF returns
#    and actual strategy weights under alternative:
#       - transaction costs: 0, 10, 25, 50 bps
#       - rebalance frequencies: daily, weekly, monthly, quarterly
# 5. Produce true-weight robustness tables where possible.
# 6. Produce transparent diagnostics when AURORA weights are missing.
#
# Important:
# - This notebook is a robustness analysis, not a new model-selection stage.
# - Primary method remains AURORA10-UAMV-B.
# - Main claim remains downside-risk control, not total-return dominance.
# - If AURORA weights are unavailable, AURORA true re-simulation cannot be
#   claimed. The notebook will still produce ROMA true sensitivity and
#   a clear missing-AURORA-weight diagnostic.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

PROJECT_CODE = "ROMA_AURORA_TWETF"
AURORA_CODE = "AURORA_TWETF"
ROMA_CODE = "ROMA_TWETF"

# Final source-aware Notebook 13B run.
NOTEBOOK13B_RUN_ID = "20260625_065916"
NOTEBOOK13B_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / PROJECT_CODE
    / "source_aware_unified_paper_comparison"
    / f"run_{NOTEBOOK13B_RUN_ID}"
)

NOTEBOOK13B_MATRIX_PATH = (
    NOTEBOOK13B_ROOT
    / "returns"
    / "notebook13B_source_aware_strict_test_return_matrix.parquet"
)

# AURORA Notebook 10 allocation run.
AURORA_NOTEBOOK10_RUN_ID = "20260624_100748"
AURORA_N10_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / AURORA_CODE
    / "uncertainty_aware_mean_variance_allocation"
    / f"run_{AURORA_NOTEBOOK10_RUN_ID}"
)

# ROMA R2 allocation run.
ROMA_R2_RUN_ID = "20260625_025314"
ROMA_R2_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / ROMA_CODE
    / "aligned_allocation_backtest"
    / f"run_{ROMA_R2_RUN_ID}"
)

ETF_RETURN_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "panels"
    / "AURORA_etf_return_panel.parquet"
)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / PROJECT_CODE
GLOBAL_TABLE_DIR = OUTPUT_ROOT / "tables"
GLOBAL_REPORT_DIR = OUTPUT_ROOT / "reports"
GLOBAL_FIGURE_DIR = OUTPUT_ROOT / "figures"
GLOBAL_MANUSCRIPT_DIR = OUTPUT_ROOT / "manuscript_assets"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = (
    OUTPUT_ROOT
    / "true_weight_transaction_cost_rebalance_sensitivity"
    / f"run_{RUN_ID}"
)

TABLE_RUN_DIR = RUN_ROOT / "tables"
RETURN_RUN_DIR = RUN_ROOT / "returns"
WEIGHT_RUN_DIR = RUN_ROOT / "weights"
FIGURE_RUN_DIR = RUN_ROOT / "figures"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
REPORT_RUN_DIR = RUN_ROOT / "reports"
MANUSCRIPT_RUN_DIR = RUN_ROOT / "manuscript_assets"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    RETURN_RUN_DIR,
    WEIGHT_RUN_DIR,
    FIGURE_RUN_DIR,
    PAPER_FIGURE_DIR,
    REPORT_RUN_DIR,
    MANUSCRIPT_RUN_DIR,
    GLOBAL_TABLE_DIR,
    GLOBAL_REPORT_DIR,
    GLOBAL_FIGURE_DIR,
    GLOBAL_MANUSCRIPT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 96)
print("Notebook 14B: True Weight Transaction-Cost and Rebalance Sensitivity")
print("=" * 96)
print("Timestamp UTC              :", RUN_TIMESTAMP)
print("Run ID                     :", RUN_ID)
print("Notebook 13B matrix        :", NOTEBOOK13B_MATRIX_PATH)
print("AURORA N10 root            :", AURORA_N10_ROOT)
print("ROMA R2 root               :", ROMA_R2_ROOT)
print("ETF return panel           :", ETF_RETURN_PANEL_PATH)
print("Run root                   :", RUN_ROOT)
print("=" * 96)

if not NOTEBOOK13B_MATRIX_PATH.exists():
    raise FileNotFoundError(f"Missing Notebook 13B matrix: {NOTEBOOK13B_MATRIX_PATH}")

if not ETF_RETURN_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing ETF return panel: {ETF_RETURN_PANEL_PATH}")

# ============================================================
# 2. Global settings
# ============================================================

ANNUALIZATION = 252

ETF_ASSETS = ["0050", "006208", "00692", "00881"]
CASH_ASSET = "CASH"
ALL_ASSETS = ETF_ASSETS + [CASH_ASSET]

PRIMARY_AURORA_POLICY = "AURORA10_UAMV_B_more60_defensive"
AURORA_POLICIES = [
    "AURORA10_UAMV_B_more60_defensive",
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    "AURORA10_validation_selected_UAMV",
]

PRIMARY_ROMA_POLICY = "ROMA_P4_balanced_regime_blend"
ROMA_POLICIES = [
    "ROMA_P4_balanced_regime_blend",
    "ROMA_P2_20d_only_return_seeking",
    "ROMA_P0_validation_selected_20_60_blend",
    "ROMA_P1_more20_return_seeking",
    "ROMA_P3_conservative_blend",
]

ROMA_BENCHMARKS = [
    "ROMA_B12_ma_timing_equal_weight",
    "ROMA_B6_00881_only",
    "ROMA_B3_0050_only",
    "ROMA_B1_equal_weight_all_etfs",
    "ROMA_B10_momentum_top2_63d",
    "ROMA_B15_minimum_variance_126d",
]

AURORA_BENCHMARKS = [
    "AURORA_B6_00881_only",
    "AURORA_B3_0050_only",
    "AURORA_B1_equal_weight_all_etfs",
]

PRIMARY_STRATEGIES = (
    AURORA_POLICIES
    + ROMA_POLICIES
    + ROMA_BENCHMARKS
    + AURORA_BENCHMARKS
)

DISPLAY_NAMES = {
    "AURORA10_UAMV_B_more60_defensive": "AURORA10-UAMV-B",
    "AURORA10_UAMV_D_low_turnover": "AURORA10-UAMV-D",
    "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA10-UAMV-E",
    "AURORA10_UAMV_A_balanced": "AURORA10-UAMV-A",
    "AURORA10_UAMV_C_more60_growth": "AURORA10-UAMV-C",
    "AURORA10_validation_selected_UAMV": "AURORA validation-selected",
    "ROMA_P4_balanced_regime_blend": "ROMA-P4",
    "ROMA_P2_20d_only_return_seeking": "ROMA-P2",
    "ROMA_P0_validation_selected_20_60_blend": "ROMA-P0",
    "ROMA_P1_more20_return_seeking": "ROMA-P1",
    "ROMA_P3_conservative_blend": "ROMA-P3",
    "ROMA_B12_ma_timing_equal_weight": "ROMA-B12 timing",
    "ROMA_B6_00881_only": "ROMA-B6 00881",
    "ROMA_B3_0050_only": "ROMA-B3 0050",
    "ROMA_B1_equal_weight_all_etfs": "ROMA-B1 equal-weight",
    "ROMA_B10_momentum_top2_63d": "ROMA-B10 momentum",
    "ROMA_B15_minimum_variance_126d": "ROMA-B15 min-var",
    "AURORA_B6_00881_only": "AURORA-B6 00881",
    "AURORA_B3_0050_only": "AURORA-B3 0050",
    "AURORA_B1_equal_weight_all_etfs": "AURORA-B1 equal-weight",
}

POLICY_GROUPS = {
    **{p: "AURORA" for p in AURORA_POLICIES},
    **{p: "ROMA baseline" for p in ROMA_POLICIES},
    **{p: "ROMA benchmark" for p in ROMA_BENCHMARKS},
    **{p: "AURORA benchmark" for p in AURORA_BENCHMARKS},
}

TRANSACTION_COST_BPS_GRID = [0, 10, 25, 50]

REBALANCE_FREQUENCIES = {
    "daily": "D",
    "weekly": "W-FRI",
    "monthly": "M",
    "quarterly": "Q",
}

# Rebalance policy:
# - daily means use each stored weight date.
# - weekly/monthly/quarterly means use the first available trading date
#   in each period and hold weights until next rebalance.

# ============================================================
# 3. Utility functions
# ============================================================

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file extension: {path}")

def safe_read_table(path: Path):
    try:
        return read_table(path)
    except Exception as e:
        print(f"Read failed: {path}: {repr(e)}")
        return None

def standardize_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        out = out.set_index("date")
    else:
        out.index = pd.to_datetime(out.index)
    out = out.sort_index()
    return out

def write_json(path: Path, obj) -> None:
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def write_markdown(path: Path, text: str) -> None:
    Path(path).write_text(text, encoding="utf-8")

def save_table(df: pd.DataFrame, local_name: str, global_name: str | None = None):
    local_path = TABLE_RUN_DIR / local_name
    df.to_csv(local_path, index=False)
    global_path = None
    if global_name is not None:
        global_path = GLOBAL_TABLE_DIR / global_name
        df.to_csv(global_path, index=False)
    return local_path, global_path

def save_parquet_csv(df: pd.DataFrame, base_path: Path, index=True):
    csv_path = Path(str(base_path) + ".csv")
    parquet_path = Path(str(base_path) + ".parquet")
    df.to_csv(csv_path, index=index)
    try:
        df.to_parquet(parquet_path, index=index)
    except Exception as e:
        print("Parquet save skipped:", parquet_path, repr(e))
        parquet_path = None
    return parquet_path, csv_path

def sha256_file(path: Path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root: Path) -> pd.DataFrame:
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def equity_from_returns(r: pd.Series) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    return (1.0 + r).cumprod()

def drawdown_from_equity(equity: pd.Series) -> pd.Series:
    equity = pd.Series(equity).astype(float)
    return equity / equity.cummax() - 1.0

def performance_metrics(r: pd.Series, annualization=ANNUALIZATION) -> dict:
    r = pd.Series(r).dropna().astype(float)
    if len(r) == 0:
        return {
            "n_days": 0,
            "start_date": pd.NaT,
            "end_date": pd.NaT,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "final_equity": np.nan,
            "mean_daily_return": np.nan,
            "daily_volatility": np.nan,
            "positive_day_rate": np.nan,
            "worst_daily_return": np.nan,
            "best_daily_return": np.nan,
        }

    r.index = pd.to_datetime(r.index)
    n = len(r)
    equity = equity_from_returns(r)
    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (annualization / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(annualization)) if np.isfinite(daily_vol) else np.nan

    mean_daily = float(r.mean())
    sharpe = float((mean_daily / daily_vol) * np.sqrt(annualization)) if daily_vol and daily_vol > 0 else np.nan

    negative = r[r < 0]
    downside_vol = float(negative.std(ddof=1)) if len(negative) > 1 else np.nan
    sortino = float((mean_daily / downside_vol) * np.sqrt(annualization)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    dd = drawdown_from_equity(equity)
    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": r.index.min(),
        "end_date": r.index.max(),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "final_equity": float(equity.iloc[-1]),
        "mean_daily_return": mean_daily,
        "daily_volatility": daily_vol,
        "positive_day_rate": float((r > 0).mean()),
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
    }

def build_performance_table(return_matrix: pd.DataFrame, extra_cols: dict | None = None) -> pd.DataFrame:
    rows = []
    for strategy in return_matrix.columns:
        m = performance_metrics(return_matrix[strategy])
        m["strategy_name"] = strategy
        m["display_name"] = DISPLAY_NAMES.get(strategy, strategy)
        m["strategy_group"] = POLICY_GROUPS.get(strategy, "other")
        if extra_cols:
            m.update(extra_cols)
        rows.append(m)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_annual_return"] = out["annual_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino"].rank(ascending=False, method="min")
    out["rank_max_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar"].rank(ascending=False, method="min")

    out["composite_rank"] = (
        out["rank_total_return"]
        + out["rank_annual_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_max_drawdown"]
        + out["rank_calmar"]
    ) / 6.0

    out = out.sort_values(
        ["composite_rank", "sharpe", "total_return"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    return out

def paired_difference(strategy_returns: pd.Series, benchmark_returns: pd.Series) -> dict:
    s = pd.Series(strategy_returns).dropna().astype(float)
    b = pd.Series(benchmark_returns).dropna().astype(float)
    common = s.index.intersection(b.index).sort_values()
    s = s.loc[common]
    b = b.loc[common]

    ps = performance_metrics(s)
    pb = performance_metrics(b)
    excess = s - b

    return {
        "n_days": int(len(common)),
        "strategy_total_return": ps["total_return"],
        "benchmark_total_return": pb["total_return"],
        "diff_total_return": ps["total_return"] - pb["total_return"],
        "strategy_annual_return": ps["annual_return"],
        "benchmark_annual_return": pb["annual_return"],
        "diff_annual_return": ps["annual_return"] - pb["annual_return"],
        "strategy_sharpe": ps["sharpe"],
        "benchmark_sharpe": pb["sharpe"],
        "diff_sharpe": ps["sharpe"] - pb["sharpe"],
        "strategy_sortino": ps["sortino"],
        "benchmark_sortino": pb["sortino"],
        "diff_sortino": ps["sortino"] - pb["sortino"],
        "strategy_max_drawdown": ps["max_drawdown"],
        "benchmark_max_drawdown": pb["max_drawdown"],
        "drawdown_improvement": ps["max_drawdown"] - pb["max_drawdown"],
        "strategy_calmar": ps["calmar"],
        "benchmark_calmar": pb["calmar"],
        "diff_calmar": ps["calmar"] - pb["calmar"],
        "annualized_mean_excess_return": float(excess.mean() * ANNUALIZATION),
        "excess_hit_rate": float((excess > 0).mean()),
    }

def canonical_strategy_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "strategy_name" not in out.columns:
        for c in ["policy_name", "strategy", "policy", "name", "portfolio_name"]:
            if c in out.columns:
                out = out.rename(columns={c: "strategy_name"})
                break
    return out

def canonical_date_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" not in out.columns:
        for c in ["Date", "datetime", "timestamp", "rebalance_date"]:
            if c in out.columns:
                out = out.rename(columns={c: "date"})
                break
    if "date" not in out.columns:
        out = out.reset_index()
        if "index" in out.columns:
            out = out.rename(columns={"index": "date"})
    out["date"] = pd.to_datetime(out["date"])
    return out

def map_strategy_name(x: str, source: str) -> str:
    x = str(x)

    # Already source-aware or policy-aware.
    if x.startswith("AURORA10_") or x.startswith("ROMA_"):
        return x

    # AURORA UAMV aliases.
    aurora_aliases = {
        "UAMV_A_balanced": "AURORA10_UAMV_A_balanced",
        "UAMV_B_more60_defensive": "AURORA10_UAMV_B_more60_defensive",
        "UAMV_C_more60_growth": "AURORA10_UAMV_C_more60_growth",
        "UAMV_D_low_turnover": "AURORA10_UAMV_D_low_turnover",
        "UAMV_E_no_regime_tilt_control": "AURORA10_UAMV_E_no_regime_tilt_control",
        "validation_selected_UAMV": "AURORA10_validation_selected_UAMV",
    }
    if x in aurora_aliases:
        return aurora_aliases[x]

    # Benchmark aliases.
    if source == "AURORA":
        if x.startswith("B"):
            return f"AURORA_{x}"
        return x

    if source == "ROMA":
        if x.startswith("B") or x.startswith("P"):
            return f"ROMA_{x}"
        return x

    return x

def standardize_weight_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename = {}

    for c in out.columns:
        lc = str(c).strip().lower()
        clean = lc.replace("weight_", "").replace("_weight", "").replace("w_", "")
        clean = clean.upper() if clean == "cash" else clean

        if clean in ["0050", "006208", "00692", "00881"]:
            rename[c] = clean
        elif clean in ["cash", "CASH"]:
            rename[c] = "CASH"

    out = out.rename(columns=rename)
    return out

def normalize_weights(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for asset in ALL_ASSETS:
        if asset not in out.columns:
            out[asset] = 0.0

    out[ALL_ASSETS] = out[ALL_ASSETS].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    weight_sum = out[ALL_ASSETS].sum(axis=1)
    nonzero = weight_sum.abs() > 1e-12
    out.loc[nonzero, ALL_ASSETS] = out.loc[nonzero, ALL_ASSETS].div(weight_sum.loc[nonzero], axis=0)

    return out

def deduplicate_weights(df: pd.DataFrame, source: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fixes the Notebook 14 duplicate-label issue.

    Deduplication priority:
    1. Filter strict-test rows if split/period/fold markers exist.
    2. Sort by date, strategy, is_rebalance if present.
    3. Group by date + strategy_name and keep the last row.
    """
    if df.empty:
        return df.copy(), pd.DataFrame()

    out = df.copy()
    original_rows = len(out)

    # Filter to strict-test / test rows if possible.
    filter_report = []

    for c in ["split", "period", "sample", "dataset_split"]:
        if c in out.columns:
            vals = out[c].astype(str).str.lower()
            preferred = vals.isin([
                "strict_test_only",
                "strict_test",
                "test",
                "all",  # AURORA sometimes stores split as all but dates are already strict-test after filtering.
            ])
            if preferred.any():
                before = len(out)
                out = out.loc[preferred].copy()
                filter_report.append({
                    "source": source,
                    "filter_column": c,
                    "before_rows": before,
                    "after_rows": len(out),
                    "kept_values": sorted(out[c].astype(str).unique().tolist()),
                })
                break

    # Sort with useful columns.
    sort_cols = ["date", "strategy_name"]
    if "is_rebalance" in out.columns:
        sort_cols.append("is_rebalance")
    if "is_rebalance_date" in out.columns:
        sort_cols.append("is_rebalance_date")
    if "fold_number" in out.columns:
        sort_cols.append("fold_number")
    if "fold_id" in out.columns:
        sort_cols.append("fold_id")

    sort_cols = [c for c in sort_cols if c in out.columns]
    out = out.sort_values(sort_cols)

    duplicate_count = int(out.duplicated(["date", "strategy_name"]).sum())

    dedup = (
        out.groupby(["date", "strategy_name"], as_index=False)
        .tail(1)
        .sort_values(["strategy_name", "date"])
        .reset_index(drop=True)
    )

    report = {
        "source": source,
        "original_rows": int(original_rows),
        "rows_after_split_filter": int(len(out)),
        "duplicate_date_strategy_rows_before_dedup": duplicate_count,
        "rows_after_dedup": int(len(dedup)),
        "unique_strategies_after_dedup": int(dedup["strategy_name"].nunique()) if "strategy_name" in dedup.columns else 0,
        "min_date": str(dedup["date"].min()) if len(dedup) else None,
        "max_date": str(dedup["date"].max()) if len(dedup) else None,
    }

    report_df = pd.DataFrame([report])
    filter_df = pd.DataFrame(filter_report)

    if len(filter_df):
        report_df = report_df.merge(filter_df, on="source", how="left")

    return dedup, report_df

def standardize_weights(raw: pd.DataFrame, source: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if raw is None or raw.empty:
        return pd.DataFrame(), pd.DataFrame([{
            "source": source,
            "status": "empty_raw_weights",
        }])

    out = raw.copy()
    out = canonical_date_column(out)
    out = canonical_strategy_column(out)

    if "strategy_name" not in out.columns:
        return pd.DataFrame(), pd.DataFrame([{
            "source": source,
            "status": "missing_strategy_name",
            "columns": ",".join(map(str, raw.columns)),
        }])

    out["strategy_name"] = out["strategy_name"].astype(str).map(lambda x: map_strategy_name(x, source))
    out = standardize_weight_columns(out)

    asset_cols = [c for c in ALL_ASSETS if c in out.columns]
    if len([a for a in ETF_ASSETS if a in asset_cols]) == 0:
        return pd.DataFrame(), pd.DataFrame([{
            "source": source,
            "status": "missing_asset_weight_columns",
            "columns": ",".join(map(str, raw.columns)),
        }])

    out = normalize_weights(out)

    keep_meta = [
        c for c in [
            "date",
            "strategy_name",
            "split",
            "period",
            "sample",
            "dataset_split",
            "is_rebalance",
            "is_rebalance_date",
            "fold_id",
            "fold_number",
        ]
        if c in out.columns
    ]

    out = out[keep_meta + ALL_ASSETS].copy()

    dedup, report = deduplicate_weights(out, source)
    report["status"] = "ok"

    return dedup, report

def rebalance_dates_from_index(index: pd.DatetimeIndex, frequency: str) -> pd.DatetimeIndex:
    index = pd.DatetimeIndex(index).sort_values()

    if frequency == "daily":
        return index

    if frequency == "weekly":
        periods = index.to_period("W-FRI")
    elif frequency == "monthly":
        periods = index.to_period("M")
    elif frequency == "quarterly":
        periods = index.to_period("Q")
    else:
        raise ValueError(f"Unknown rebalance frequency: {frequency}")

    first_dates = pd.Series(index=index, data=index).groupby(periods).first()
    return pd.DatetimeIndex(first_dates.values).sort_values()

def simulate_from_weights(
    strategy_weights: pd.DataFrame,
    strategy_name: str,
    etf_returns: pd.DataFrame,
    dates: pd.DatetimeIndex,
    rebalance_frequency: str,
    transaction_cost_bps: float,
) -> pd.DataFrame:
    """
    True weight-based portfolio simulation.

    Assumptions:
    - Weights are target weights known at each date.
    - At rebalance dates, the target weight is adopted.
    - Between rebalance dates, weights are held fixed.
    - Cash return is 0.
    - Transaction cost = turnover * tc_rate.
    """
    dates = pd.DatetimeIndex(dates).sort_values()

    w = strategy_weights[strategy_weights["strategy_name"] == strategy_name].copy()
    if w.empty:
        raise ValueError(f"No weights found for {strategy_name}")

    w["date"] = pd.to_datetime(w["date"])
    w = w.sort_values("date")
    w = w.drop_duplicates(["date", "strategy_name"], keep="last")
    w = w.set_index("date")

    # Important: deduplicate index again before reindexing.
    w = w[~w.index.duplicated(keep="last")]

    for asset in ALL_ASSETS:
        if asset not in w.columns:
            w[asset] = 0.0

    # Align to dates. Forward-fill target weights.
    target_w = w[ALL_ASSETS].reindex(dates).ffill().bfill()
    target_w = normalize_weights(target_w.reset_index().rename(columns={"index": "date"})).set_index("date")
    target_w = target_w[ALL_ASSETS]

    # Determine rebalance dates.
    rb_dates = rebalance_dates_from_index(dates, rebalance_frequency)
    rb_mask = pd.Series(False, index=dates)
    rb_mask.loc[rb_dates] = True

    held_w = pd.DataFrame(index=dates, columns=ALL_ASSETS, dtype=float)

    current_w = None
    for dt in dates:
        if current_w is None:
            current_w = target_w.loc[dt].copy()
        elif bool(rb_mask.loc[dt]):
            current_w = target_w.loc[dt].copy()
        held_w.loc[dt, ALL_ASSETS] = current_w.values

    held_w = held_w.astype(float).fillna(0.0)
    held_w = normalize_weights(held_w.reset_index().rename(columns={"index": "date"})).set_index("date")
    held_w = held_w[ALL_ASSETS]

    # Align return panel.
    ret = etf_returns.copy()
    ret.index = pd.to_datetime(ret.index)
    ret = ret.sort_index()
    ret = ret.reindex(dates)

    for asset in ETF_ASSETS:
        if asset not in ret.columns:
            raise ValueError(f"ETF return column missing: {asset}")

    asset_returns = ret[ETF_ASSETS].fillna(0.0)

    gross_return = (held_w[ETF_ASSETS].values * asset_returns[ETF_ASSETS].values).sum(axis=1)
    gross_return = pd.Series(gross_return, index=dates)

    # Turnover at rebalances: sum absolute change in all asset weights.
    prev_w = held_w.shift(1).fillna(0.0)
    turnover = (held_w[ALL_ASSETS] - prev_w[ALL_ASSETS]).abs().sum(axis=1)
    turnover.loc[~rb_mask] = 0.0

    tc_rate = float(transaction_cost_bps) / 10000.0
    transaction_cost = turnover * tc_rate
    net_return = gross_return - transaction_cost

    out = pd.DataFrame({
        "date": dates,
        "strategy_name": strategy_name,
        "rebalance_frequency": rebalance_frequency,
        "transaction_cost_bps": transaction_cost_bps,
        "gross_return": gross_return.values,
        "turnover": turnover.values,
        "transaction_cost": transaction_cost.values,
        "net_return": net_return.values,
        "is_rebalance_date": rb_mask.values,
    })

    for asset in ALL_ASSETS:
        out[f"weight_{asset}"] = held_w[asset].values

    out["equity"] = (1.0 + out["net_return"]).cumprod()
    out["drawdown"] = out["equity"] / out["equity"].cummax() - 1.0

    return out

# ============================================================
# 4. Load Notebook 13B return matrix and ETF returns
# ============================================================

print("\n" + "=" * 96)
print("Step 1: Loading source-aware returns and ETF returns")
print("=" * 96)

source_mat = standardize_date_index(read_table(NOTEBOOK13B_MATRIX_PATH))
source_mat = source_mat.sort_index()

etf_returns = standardize_date_index(read_table(ETF_RETURN_PANEL_PATH))
etf_returns = etf_returns.sort_index()

strict_dates = source_mat.index
etf_returns = etf_returns.reindex(strict_dates)

print("Source-aware matrix:", source_mat.shape, source_mat.index.min(), "to", source_mat.index.max())
print("ETF returns        :", etf_returns.shape, etf_returns.index.min(), "to", etf_returns.index.max())
print("ETF return columns :", etf_returns.columns.tolist())

base_strategies_available = [s for s in PRIMARY_STRATEGIES if s in source_mat.columns]
base_mat = source_mat[base_strategies_available].copy()

base_perf = build_performance_table(
    base_mat,
    extra_cols={
        "source": "Notebook13B_original_return_matrix",
        "rebalance_frequency": "as_generated",
        "transaction_cost_bps": "as_generated",
    },
)

save_table(
    base_perf,
    "notebook14B_00_original_notebook13B_performance.csv",
    f"table_14B_00_original_notebook13B_performance_{RUN_ID}.csv",
)

# ============================================================
# 5. Auto-discover weight files
# ============================================================

print("\n" + "=" * 96)
print("Step 2: Auto-discovering AURORA and ROMA weight files")
print("=" * 96)

def discover_weight_files(root: Path, source: str) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame()

    patterns = ["*weight*.parquet", "*weights*.parquet", "*weight*.csv", "*weights*.csv"]
    files = []
    for pat in patterns:
        files.extend(list(root.rglob(pat)))

    files = sorted(set(files))

    for p in files:
        try:
            df = read_table(p)
            cols = list(map(str, df.columns))
            has_date = any(c.lower() in ["date", "datetime", "timestamp", "rebalance_date"] for c in cols)
            has_strategy = any(c.lower() in ["strategy_name", "policy_name", "strategy", "policy", "name", "portfolio_name"] for c in cols)
            lower_cols = [c.lower() for c in cols]
            has_asset = any(asset in lower_cols for asset in ["0050", "006208", "00692", "00881"]) or any(
                asset in c for c in lower_cols for asset in ["0050", "006208", "00692", "00881"]
            )
            rows.append({
                "source": source,
                "path": str(p),
                "shape": str(df.shape),
                "n_rows": int(df.shape[0]),
                "n_cols": int(df.shape[1]),
                "has_date_like": has_date,
                "has_strategy_like": has_strategy,
                "has_asset_like": has_asset,
                "columns_preview": ", ".join(cols[:30]),
                "score": int(has_date) + int(has_strategy) + int(has_asset) + min(int(df.shape[0] > 0), 1),
            })
        except Exception as e:
            rows.append({
                "source": source,
                "path": str(p),
                "shape": "read_failed",
                "n_rows": 0,
                "n_cols": 0,
                "has_date_like": False,
                "has_strategy_like": False,
                "has_asset_like": False,
                "columns_preview": repr(e),
                "score": -1,
            })

    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["score", "n_rows"], ascending=[False, False]).reset_index(drop=True)
    return out

aurora_weight_candidates = discover_weight_files(AURORA_N10_ROOT, "AURORA")
roma_weight_candidates = discover_weight_files(ROMA_R2_ROOT, "ROMA")

save_table(
    aurora_weight_candidates,
    "notebook14B_01_aurora_weight_file_candidates.csv",
    f"table_14B_01_aurora_weight_file_candidates_{RUN_ID}.csv",
)

save_table(
    roma_weight_candidates,
    "notebook14B_02_roma_weight_file_candidates.csv",
    f"table_14B_02_roma_weight_file_candidates_{RUN_ID}.csv",
)

print("Top AURORA weight candidates:")
if len(aurora_weight_candidates):
    print(aurora_weight_candidates.head(10).to_string(index=False))
else:
    print("No AURORA weight candidates found.")

print("\nTop ROMA weight candidates:")
if len(roma_weight_candidates):
    print(roma_weight_candidates.head(10).to_string(index=False))
else:
    print("No ROMA weight candidates found.")

def pick_best_weight_file(candidates: pd.DataFrame):
    if candidates is None or candidates.empty:
        return None
    good = candidates[
        (candidates["score"] >= 3)
        & (candidates["has_asset_like"] == True)
    ].copy()
    if good.empty:
        return None
    return Path(good.iloc[0]["path"])

aurora_weight_path = pick_best_weight_file(aurora_weight_candidates)
roma_weight_path = pick_best_weight_file(roma_weight_candidates)

print("\nSelected AURORA weight path:", aurora_weight_path)
print("Selected ROMA weight path  :", roma_weight_path)

# ============================================================
# 6. Load, standardize, deduplicate weights
# ============================================================

print("\n" + "=" * 96)
print("Step 3: Loading, standardizing, and deduplicating weights")
print("=" * 96)

if aurora_weight_path is not None:
    aurora_raw = read_table(aurora_weight_path)
else:
    aurora_raw = pd.DataFrame()

if roma_weight_path is not None:
    roma_raw = read_table(roma_weight_path)
else:
    roma_raw = pd.DataFrame()

aurora_weights, aurora_weight_report = standardize_weights(aurora_raw, "AURORA")
roma_weights, roma_weight_report = standardize_weights(roma_raw, "ROMA")

all_weight_reports = pd.concat(
    [aurora_weight_report, roma_weight_report],
    axis=0,
    ignore_index=True,
)

save_table(
    all_weight_reports,
    "notebook14B_03_weight_standardization_report.csv",
    f"table_14B_03_weight_standardization_report_{RUN_ID}.csv",
)

# Restrict to strict-test dates.
def restrict_weights_to_strict_dates(w: pd.DataFrame) -> pd.DataFrame:
    if w.empty:
        return w
    out = w.copy()
    out["date"] = pd.to_datetime(out["date"])
    out = out[
        (out["date"] >= strict_dates.min())
        & (out["date"] <= strict_dates.max())
    ].copy()
    out = out.sort_values(["strategy_name", "date"]).reset_index(drop=True)
    return out

aurora_weights = restrict_weights_to_strict_dates(aurora_weights)
roma_weights = restrict_weights_to_strict_dates(roma_weights)

all_weights = pd.concat([aurora_weights, roma_weights], axis=0, ignore_index=True)

save_parquet_csv(
    aurora_weights,
    WEIGHT_RUN_DIR / "notebook14B_aurora_weights_standardized_deduplicated_strict",
    index=False,
)

save_parquet_csv(
    roma_weights,
    WEIGHT_RUN_DIR / "notebook14B_roma_weights_standardized_deduplicated_strict",
    index=False,
)

save_parquet_csv(
    all_weights,
    WEIGHT_RUN_DIR / "notebook14B_all_weights_standardized_deduplicated_strict",
    index=False,
)

save_table(
    all_weights,
    "notebook14B_04_all_weights_standardized_deduplicated_strict.csv",
    f"table_14B_04_all_weights_standardized_deduplicated_strict_{RUN_ID}.csv",
)

print("Weight standardization report:")
print(all_weight_reports.to_string(index=False))

print("\nAURORA standardized weights:", aurora_weights.shape)
if len(aurora_weights):
    print(sorted(aurora_weights["strategy_name"].unique()))
else:
    print("No usable AURORA weights found.")

print("\nROMA standardized weights:", roma_weights.shape)
if len(roma_weights):
    print(sorted(roma_weights["strategy_name"].unique()))
else:
    print("No usable ROMA weights found.")

# ============================================================
# 7. Coverage audit
# ============================================================

print("\n" + "=" * 96)
print("Step 4: Weight coverage audit")
print("=" * 96)

coverage_rows = []

for strategy in PRIMARY_STRATEGIES:
    in_13b = strategy in source_mat.columns
    in_weights = len(all_weights[all_weights["strategy_name"] == strategy]) > 0 if len(all_weights) else False

    row = {
        "strategy_name": strategy,
        "display_name": DISPLAY_NAMES.get(strategy, strategy),
        "strategy_group": POLICY_GROUPS.get(strategy, "other"),
        "available_in_notebook13B_matrix": in_13b,
        "available_in_standardized_weights": in_weights,
    }

    if in_weights:
        sub = all_weights[all_weights["strategy_name"] == strategy].copy()
        row.update({
            "n_weight_dates": int(sub["date"].nunique()),
            "min_weight_date": str(sub["date"].min().date()),
            "max_weight_date": str(sub["date"].max().date()),
            "duplicate_date_strategy_after_dedup": int(sub.duplicated(["date", "strategy_name"]).sum()),
            "mean_weight_sum": float(sub[ALL_ASSETS].sum(axis=1).mean()),
            "min_weight_sum": float(sub[ALL_ASSETS].sum(axis=1).min()),
            "max_weight_sum": float(sub[ALL_ASSETS].sum(axis=1).max()),
        })

    coverage_rows.append(row)

coverage_audit = pd.DataFrame(coverage_rows)

save_table(
    coverage_audit,
    "notebook14B_05_weight_coverage_audit.csv",
    f"table_14B_05_weight_coverage_audit_{RUN_ID}.csv",
)

print("Coverage audit:")
print(coverage_audit.to_string(index=False))

# ============================================================
# 8. True weight-based simulation grid
# ============================================================

print("\n" + "=" * 96)
print("Step 5: Running true weight-based simulation grid")
print("=" * 96)

strategies_with_weights = [
    s for s in PRIMARY_STRATEGIES
    if len(all_weights[all_weights["strategy_name"] == s]) > 0
]

print("Strategies with usable weights:")
for s in strategies_with_weights:
    print(" -", s)

simulation_rows = []
return_matrices = {}
simulation_error_rows = []

for freq in REBALANCE_FREQUENCIES.keys():
    for tc_bps in TRANSACTION_COST_BPS_GRID:
        scenario_key = f"{freq}_{tc_bps}bps"
        mat = pd.DataFrame(index=strict_dates)

        for strategy in strategies_with_weights:
            try:
                sim = simulate_from_weights(
                    strategy_weights=all_weights,
                    strategy_name=strategy,
                    etf_returns=etf_returns,
                    dates=strict_dates,
                    rebalance_frequency=freq,
                    transaction_cost_bps=tc_bps,
                )

                sim_base = RETURN_RUN_DIR / f"notebook14B_true_sim_{strategy}_{scenario_key}"
                save_parquet_csv(sim, sim_base, index=False)

                r = sim.set_index("date")["net_return"]
                mat[strategy] = r.reindex(strict_dates)

            except Exception as e:
                simulation_error_rows.append({
                    "rebalance_frequency": freq,
                    "transaction_cost_bps": tc_bps,
                    "scenario_key": scenario_key,
                    "strategy_name": strategy,
                    "error": repr(e),
                })
                print(f"Simulation failed: {strategy}, {scenario_key}: {repr(e)}")

        if len(mat.columns):
            return_matrices[scenario_key] = mat.copy()
            save_parquet_csv(
                mat,
                RETURN_RUN_DIR / f"notebook14B_true_weight_return_matrix_{scenario_key}",
                index=True,
            )

            perf = build_performance_table(
                mat,
                extra_cols={
                    "sensitivity_source": "true_weight_resimulation",
                    "rebalance_frequency": freq,
                    "transaction_cost_bps": tc_bps,
                    "scenario_key": scenario_key,
                },
            )
            simulation_rows.append(perf)

if simulation_rows:
    true_perf = pd.concat(simulation_rows, axis=0, ignore_index=True)
else:
    true_perf = pd.DataFrame()

simulation_errors = pd.DataFrame(simulation_error_rows)

save_table(
    true_perf,
    "notebook14B_06_true_weight_sensitivity_performance.csv",
    f"table_14B_06_true_weight_sensitivity_performance_{RUN_ID}.csv",
)

save_table(
    simulation_errors,
    "notebook14B_07_true_weight_simulation_errors.csv",
    f"table_14B_07_true_weight_simulation_errors_{RUN_ID}.csv",
)

print("True weight performance shape:", true_perf.shape)
print("Simulation errors shape      :", simulation_errors.shape)

# ============================================================
# 9. Original-matrix fallback for missing AURORA only
# ============================================================

print("\n" + "=" * 96)
print("Step 6: Creating transparent fallback diagnostics for missing strategies")
print("=" * 96)

missing_weight_strategies = [
    s for s in PRIMARY_STRATEGIES
    if (s in source_mat.columns) and (s not in strategies_with_weights)
]

print("Strategies available in 13B but missing true weights:")
for s in missing_weight_strategies:
    print(" -", s)

missing_weight_diag = pd.DataFrame([
    {
        "strategy_name": s,
        "display_name": DISPLAY_NAMES.get(s, s),
        "available_in_notebook13B_matrix": s in source_mat.columns,
        "available_in_true_weights": s in strategies_with_weights,
        "can_true_resimulate": s in strategies_with_weights,
        "diagnostic_action": (
            "Cannot true re-simulate without exported daily weights; "
            "use original Notebook 13B return only for non-robustness reference."
        ),
    }
    for s in missing_weight_strategies
])

save_table(
    missing_weight_diag,
    "notebook14B_08_missing_weight_strategy_diagnostic.csv",
    f"table_14B_08_missing_weight_strategy_diagnostic_{RUN_ID}.csv",
)

# Non-robust reference performance for missing strategies.
if missing_weight_strategies:
    missing_reference_mat = source_mat[missing_weight_strategies].copy()
    missing_reference_perf = build_performance_table(
        missing_reference_mat,
        extra_cols={
            "sensitivity_source": "not_true_resimulation_original_13B_reference_only",
            "rebalance_frequency": "as_generated",
            "transaction_cost_bps": "as_generated",
            "scenario_key": "original_13B_reference_only",
        },
    )
else:
    missing_reference_perf = pd.DataFrame()

save_table(
    missing_reference_perf,
    "notebook14B_09_missing_weight_original_reference_performance.csv",
    f"table_14B_09_missing_weight_original_reference_performance_{RUN_ID}.csv",
)

# ============================================================
# 10. Pairwise robustness comparisons
# ============================================================

print("\n" + "=" * 96)
print("Step 7: Pairwise true-weight robustness comparisons")
print("=" * 96)

PAIRWISE_SPECS = [
    (PRIMARY_AURORA_POLICY, PRIMARY_ROMA_POLICY, "AURORA10-UAMV-B vs ROMA-P4"),
    (PRIMARY_AURORA_POLICY, "ROMA_P2_20d_only_return_seeking", "AURORA10-UAMV-B vs ROMA-P2"),
    (PRIMARY_AURORA_POLICY, "ROMA_B12_ma_timing_equal_weight", "AURORA10-UAMV-B vs ROMA-B12"),
    (PRIMARY_AURORA_POLICY, "ROMA_B6_00881_only", "AURORA10-UAMV-B vs ROMA-B6"),
    (PRIMARY_AURORA_POLICY, "ROMA_B3_0050_only", "AURORA10-UAMV-B vs ROMA-B3"),
    (PRIMARY_AURORA_POLICY, "ROMA_B1_equal_weight_all_etfs", "AURORA10-UAMV-B vs ROMA-B1"),
    (PRIMARY_ROMA_POLICY, "ROMA_B12_ma_timing_equal_weight", "ROMA-P4 vs ROMA-B12"),
    (PRIMARY_ROMA_POLICY, "ROMA_B6_00881_only", "ROMA-P4 vs ROMA-B6"),
    (PRIMARY_ROMA_POLICY, "ROMA_B3_0050_only", "ROMA-P4 vs ROMA-B3"),
    (PRIMARY_ROMA_POLICY, "ROMA_B1_equal_weight_all_etfs", "ROMA-P4 vs ROMA-B1"),
]

pairwise_rows = []

for scenario_key, mat in return_matrices.items():
    freq, tc_part = scenario_key.rsplit("_", 1)
    tc_bps = float(tc_part.replace("bps", ""))

    for strategy, benchmark, label in PAIRWISE_SPECS:
        if strategy in mat.columns and benchmark in mat.columns:
            diff = paired_difference(mat[strategy], mat[benchmark])
            pairwise_rows.append({
                "sensitivity_source": "true_weight_resimulation",
                "scenario_key": scenario_key,
                "rebalance_frequency": freq,
                "transaction_cost_bps": tc_bps,
                "comparison_label": label,
                "strategy_name": strategy,
                "strategy_display_name": DISPLAY_NAMES.get(strategy, strategy),
                "benchmark_name": benchmark,
                "benchmark_display_name": DISPLAY_NAMES.get(benchmark, benchmark),
                **diff,
            })
        else:
            pairwise_rows.append({
                "sensitivity_source": "true_weight_resimulation",
                "scenario_key": scenario_key,
                "rebalance_frequency": freq,
                "transaction_cost_bps": tc_bps,
                "comparison_label": label,
                "strategy_name": strategy,
                "strategy_display_name": DISPLAY_NAMES.get(strategy, strategy),
                "benchmark_name": benchmark,
                "benchmark_display_name": DISPLAY_NAMES.get(benchmark, benchmark),
                "missing_reason": "strategy_or_benchmark_weight_not_available",
            })

pairwise_df = pd.DataFrame(pairwise_rows)

save_table(
    pairwise_df,
    "notebook14B_10_true_weight_pairwise_robustness.csv",
    f"table_14B_10_true_weight_pairwise_robustness_{RUN_ID}.csv",
)

print("Pairwise true-weight robustness shape:", pairwise_df.shape)
if len(pairwise_df):
    preview_cols = [
        "rebalance_frequency",
        "transaction_cost_bps",
        "comparison_label",
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
        "diff_calmar",
        "missing_reason",
    ]
    preview_cols = [c for c in preview_cols if c in pairwise_df.columns]
    print(pairwise_df[preview_cols].head(80).to_string(index=False))

# ============================================================
# 11. Claim stability summary
# ============================================================

print("\n" + "=" * 96)
print("Step 8: Claim stability summary")
print("=" * 96)

claim_rows = []

def perf_lookup(perf_df, strategy, freq, tc_bps):
    if perf_df.empty:
        return None
    rows = perf_df[
        (perf_df["strategy_name"] == strategy)
        & (perf_df["rebalance_frequency"] == freq)
        & (perf_df["transaction_cost_bps"].astype(float) == float(tc_bps))
    ]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

for freq in REBALANCE_FREQUENCIES.keys():
    for tc_bps in TRANSACTION_COST_BPS_GRID:
        aur = perf_lookup(true_perf, PRIMARY_AURORA_POLICY, freq, tc_bps)
        roma = perf_lookup(true_perf, PRIMARY_ROMA_POLICY, freq, tc_bps)
        b12 = perf_lookup(true_perf, "ROMA_B12_ma_timing_equal_weight", freq, tc_bps)

        row = {
            "sensitivity_source": "true_weight_resimulation",
            "rebalance_frequency": freq,
            "transaction_cost_bps": tc_bps,
            "has_aurora_true_weights": aur is not None,
            "has_roma_p4_true_weights": roma is not None,
            "has_roma_b12_true_weights": b12 is not None,
        }

        if aur is not None:
            row.update({
                "aurora_total_return": aur["total_return"],
                "aurora_sharpe": aur["sharpe"],
                "aurora_sortino": aur["sortino"],
                "aurora_max_drawdown": aur["max_drawdown"],
                "aurora_calmar": aur["calmar"],
            })

        if roma is not None:
            row.update({
                "roma_p4_total_return": roma["total_return"],
                "roma_p4_sharpe": roma["sharpe"],
                "roma_p4_sortino": roma["sortino"],
                "roma_p4_max_drawdown": roma["max_drawdown"],
                "roma_p4_calmar": roma["calmar"],
            })

        if b12 is not None:
            row.update({
                "roma_b12_total_return": b12["total_return"],
                "roma_b12_sharpe": b12["sharpe"],
                "roma_b12_sortino": b12["sortino"],
                "roma_b12_max_drawdown": b12["max_drawdown"],
                "roma_b12_calmar": b12["calmar"],
            })

        if aur is not None and roma is not None:
            row.update({
                "aurora_sharpe_gt_roma": aur["sharpe"] > roma["sharpe"],
                "aurora_sortino_gt_roma": aur["sortino"] > roma["sortino"],
                "aurora_mdd_less_severe_than_roma": aur["max_drawdown"] > roma["max_drawdown"],
                "aurora_total_return_gt_roma": aur["total_return"] > roma["total_return"],
                "aurora_minus_roma_sharpe": aur["sharpe"] - roma["sharpe"],
                "aurora_minus_roma_sortino": aur["sortino"] - roma["sortino"],
                "aurora_drawdown_improvement_vs_roma": aur["max_drawdown"] - roma["max_drawdown"],
                "aurora_minus_roma_total_return": aur["total_return"] - roma["total_return"],
            })

        if roma is not None and b12 is not None:
            row.update({
                "roma_sharpe_gt_b12": roma["sharpe"] > b12["sharpe"],
                "roma_mdd_less_severe_than_b12": roma["max_drawdown"] > b12["max_drawdown"],
                "roma_total_return_gt_b12": roma["total_return"] > b12["total_return"],
                "roma_minus_b12_sharpe": roma["sharpe"] - b12["sharpe"],
                "roma_drawdown_improvement_vs_b12": roma["max_drawdown"] - b12["max_drawdown"],
                "roma_minus_b12_total_return": roma["total_return"] - b12["total_return"],
            })

        claim_rows.append(row)

claim_by_scenario = pd.DataFrame(claim_rows)

bool_cols = [
    "aurora_sharpe_gt_roma",
    "aurora_sortino_gt_roma",
    "aurora_mdd_less_severe_than_roma",
    "aurora_total_return_gt_roma",
    "roma_sharpe_gt_b12",
    "roma_mdd_less_severe_than_b12",
    "roma_total_return_gt_b12",
]

summary_rows = []
for c in bool_cols:
    if c in claim_by_scenario.columns:
        valid = claim_by_scenario[c].dropna()
        summary_rows.append({
            "claim_indicator": c,
            "n_scenarios_available": int(len(valid)),
            "n_true": int(valid.sum()) if len(valid) else 0,
            "share_true": float(valid.mean()) if len(valid) else np.nan,
            "interpretation": (
                "No true AURORA-vs-ROMA claim possible if n_scenarios_available=0."
                if c.startswith("aurora") and len(valid) == 0
                else ""
            ),
        })

claim_summary = pd.DataFrame(summary_rows)

save_table(
    claim_by_scenario,
    "notebook14B_11_claim_stability_by_scenario.csv",
    f"table_14B_11_claim_stability_by_scenario_{RUN_ID}.csv",
)

save_table(
    claim_summary,
    "notebook14B_12_claim_stability_summary.csv",
    f"table_14B_12_claim_stability_summary_{RUN_ID}.csv",
)

print("Claim stability summary:")
print(claim_summary.to_string(index=False))

# ============================================================
# 12. Paper-ready tables
# ============================================================

print("\n" + "=" * 96)
print("Step 9: Paper-ready robustness tables")
print("=" * 96)

paper_perf = true_perf.copy()

if len(paper_perf):
    paper_perf = paper_perf[
        paper_perf["strategy_name"].isin([
            PRIMARY_AURORA_POLICY,
            "AURORA10_UAMV_D_low_turnover",
            "AURORA10_UAMV_E_no_regime_tilt_control",
            PRIMARY_ROMA_POLICY,
            "ROMA_P2_20d_only_return_seeking",
            "ROMA_B12_ma_timing_equal_weight",
            "ROMA_B6_00881_only",
            "ROMA_B3_0050_only",
            "ROMA_B1_equal_weight_all_etfs",
        ])
    ].copy()

    for c in [
        "total_return",
        "annual_return",
        "annual_volatility",
        "sharpe",
        "sortino",
        "max_drawdown",
        "calmar",
        "composite_rank",
    ]:
        if c in paper_perf.columns:
            paper_perf[c] = paper_perf[c].astype(float).round(4)

paper_pairwise = pairwise_df.copy()
if len(paper_pairwise):
    keep_labels = [
        "AURORA10-UAMV-B vs ROMA-P4",
        "AURORA10-UAMV-B vs ROMA-P2",
        "AURORA10-UAMV-B vs ROMA-B12",
        "ROMA-P4 vs ROMA-B12",
    ]
    paper_pairwise = paper_pairwise[paper_pairwise["comparison_label"].isin(keep_labels)].copy()

    for c in [
        "diff_total_return",
        "diff_annual_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
        "diff_calmar",
        "annualized_mean_excess_return",
    ]:
        if c in paper_pairwise.columns:
            paper_pairwise[c] = pd.to_numeric(paper_pairwise[c], errors="coerce").round(4)

save_table(
    paper_perf,
    "notebook14B_13_paper_true_weight_robustness_performance.csv",
    f"table_14B_13_paper_true_weight_robustness_performance_{RUN_ID}.csv",
)

save_table(
    paper_pairwise,
    "notebook14B_14_paper_true_weight_robustness_pairwise.csv",
    f"table_14B_14_paper_true_weight_robustness_pairwise_{RUN_ID}.csv",
)

print("Paper performance preview:")
if len(paper_perf):
    print(
        paper_perf[
            [
                "rebalance_frequency",
                "transaction_cost_bps",
                "display_name",
                "total_return",
                "sharpe",
                "sortino",
                "max_drawdown",
                "calmar",
            ]
        ].head(80).to_string(index=False)
    )
else:
    print("No true weight performance rows available.")

print("\nPaper pairwise preview:")
if len(paper_pairwise):
    preview_cols = [
        "rebalance_frequency",
        "transaction_cost_bps",
        "comparison_label",
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
        "diff_calmar",
        "missing_reason",
    ]
    preview_cols = [c for c in preview_cols if c in paper_pairwise.columns]
    print(paper_pairwise[preview_cols].head(80).to_string(index=False))
else:
    print("No true weight pairwise rows available.")

# ============================================================
# 13. Figures
# ============================================================

print("\n" + "=" * 96)
print("Step 10: Creating figures")
print("=" * 96)

figure_records = []

def record_figure(path, figure_id, title, caption):
    figure_records.append({
        "figure_id": figure_id,
        "path": str(path),
        "title": title,
        "caption": caption,
    })

# Figure 14B.1: ROMA-P4 vs ROMA-B12 true sensitivity heatmap.
roma_pair = pairwise_df[
    pairwise_df["comparison_label"] == "ROMA-P4 vs ROMA-B12"
].copy()

if len(roma_pair) and "diff_sharpe" in roma_pair.columns:
    pivot = roma_pair.pivot_table(
        index="rebalance_frequency",
        columns="transaction_cost_bps",
        values="diff_sharpe",
        aggfunc="first",
    )
    pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

    plt.figure(figsize=(8, 5))
    if HAS_SEABORN:
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu", center=0)
    else:
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(pivot.columns)), pivot.columns)
        plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title("ROMA-P4 minus ROMA-B12 Sharpe, true-weight sensitivity")
    plt.xlabel("Transaction cost, bps")
    plt.ylabel("Rebalance frequency")
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure14B_01_roma_minus_b12_sharpe_heatmap.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    record_figure(
        fig_path,
        "Figure 14B.1",
        "ROMA-P4 versus ROMA-B12 Sharpe sensitivity",
        "True-weight sensitivity heatmap for ROMA-P4 minus ROMA-B12 Sharpe. Positive values favor ROMA-P4.",
    )

# Figure 14B.2: ROMA-P4 vs ROMA-B12 drawdown improvement heatmap.
if len(roma_pair) and "drawdown_improvement" in roma_pair.columns:
    pivot = roma_pair.pivot_table(
        index="rebalance_frequency",
        columns="transaction_cost_bps",
        values="drawdown_improvement",
        aggfunc="first",
    )
    pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

    plt.figure(figsize=(8, 5))
    if HAS_SEABORN:
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu", center=0)
    else:
        plt.imshow(pivot.values, aspect="auto")
        plt.colorbar()
        plt.xticks(range(len(pivot.columns)), pivot.columns)
        plt.yticks(range(len(pivot.index)), pivot.index)
    plt.title("ROMA-P4 minus ROMA-B12 drawdown improvement")
    plt.xlabel("Transaction cost, bps")
    plt.ylabel("Rebalance frequency")
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure14B_02_roma_minus_b12_drawdown_heatmap.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    record_figure(
        fig_path,
        "Figure 14B.2",
        "ROMA-P4 versus ROMA-B12 drawdown sensitivity",
        "True-weight sensitivity heatmap for ROMA-P4 drawdown improvement relative to ROMA-B12. Positive values favor ROMA-P4.",
    )

# Figure 14B.3: AURORA-vs-ROMA heatmap if AURORA weights exist.
aurora_pair = pairwise_df[
    pairwise_df["comparison_label"] == "AURORA10-UAMV-B vs ROMA-P4"
].copy()

if len(aurora_pair) and "diff_sharpe" in aurora_pair.columns:
    pivot = aurora_pair.pivot_table(
        index="rebalance_frequency",
        columns="transaction_cost_bps",
        values="diff_sharpe",
        aggfunc="first",
    )
    pivot = pivot.reindex(list(REBALANCE_FREQUENCIES.keys()))

    if pivot.notna().any().any():
        plt.figure(figsize=(8, 5))
        if HAS_SEABORN:
            sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu", center=0)
        else:
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar()
            plt.xticks(range(len(pivot.columns)), pivot.columns)
            plt.yticks(range(len(pivot.index)), pivot.index)
        plt.title("AURORA10-UAMV-B minus ROMA-P4 Sharpe")
        plt.xlabel("Transaction cost, bps")
        plt.ylabel("Rebalance frequency")
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure14B_03_aurora_minus_roma_sharpe_heatmap.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            "Figure 14B.3",
            "AURORA versus ROMA-P4 Sharpe sensitivity",
            "True-weight sensitivity heatmap for AURORA10-UAMV-B minus ROMA-P4 Sharpe. Positive values favor AURORA.",
        )

# Figure 14B.4: Equity curves for a representative scenario.
scenario_for_equity = "monthly_25bps"
if scenario_for_equity not in return_matrices and len(return_matrices):
    scenario_for_equity = list(return_matrices.keys())[0]

if scenario_for_equity in return_matrices:
    mat = return_matrices[scenario_for_equity]
    plot_cols = [
        PRIMARY_AURORA_POLICY,
        PRIMARY_ROMA_POLICY,
        "ROMA_B12_ma_timing_equal_weight",
        "ROMA_B6_00881_only",
        "ROMA_B1_equal_weight_all_etfs",
    ]
    plot_cols = [c for c in plot_cols if c in mat.columns]

    if plot_cols:
        plt.figure(figsize=(11, 6))
        for c in plot_cols:
            eq = equity_from_returns(mat[c])
            plt.plot(eq.index, eq.values, linewidth=1.8, label=DISPLAY_NAMES.get(c, c))

        plt.title(f"True-weight robustness equity curves: {scenario_for_equity}")
        plt.xlabel("Date")
        plt.ylabel("Equity")
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=9)
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / f"figure14B_04_true_weight_equity_{scenario_for_equity}.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            "Figure 14B.4",
            f"True-weight equity curves under {scenario_for_equity}",
            "Equity curves for available strategies under a representative true-weight transaction-cost and rebalance-frequency scenario.",
        )

# Figure 14B.5: Claim stability if available.
if len(claim_summary):
    plot_df = claim_summary.copy()
    plot_df = plot_df.dropna(subset=["share_true"])

    if len(plot_df):
        plt.figure(figsize=(10, 5))
        plt.barh(plot_df["claim_indicator"], plot_df["share_true"], color="#4C78A8")
        plt.xlabel("Share of available true-weight scenarios")
        plt.xlim(0, 1)
        plt.title("True-weight claim stability")
        plt.grid(axis="x", alpha=0.3)
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure14B_05_true_weight_claim_stability.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            "Figure 14B.5",
            "True-weight claim stability",
            "Share of available true-weight robustness scenarios in which each claim indicator holds.",
        )

figure_index = pd.DataFrame(figure_records)

save_table(
    figure_index,
    "notebook14B_15_figure_index.csv",
    f"table_14B_15_figure_index_{RUN_ID}.csv",
)

print("Figure index:")
if len(figure_index):
    print(figure_index.to_string(index=False))
else:
    print("No figures created.")

# ============================================================
# 14. Manuscript text assets
# ============================================================

print("\n" + "=" * 96)
print("Step 11: Writing manuscript assets")
print("=" * 96)

has_aurora_weights = PRIMARY_AURORA_POLICY in strategies_with_weights
has_roma_weights = PRIMARY_ROMA_POLICY in strategies_with_weights
has_b12_weights = "ROMA_B12_ma_timing_equal_weight" in strategies_with_weights

def share_line(indicator: str) -> str:
    if claim_summary.empty:
        return "not available"
    rows = claim_summary[claim_summary["claim_indicator"] == indicator]
    if rows.empty:
        return "not available"
    r = rows.iloc[0]
    if pd.isna(r["share_true"]):
        return "not available"
    return f"{int(r['n_true'])}/{int(r['n_scenarios_available'])} scenarios ({100*float(r['share_true']):.1f}%)"

methods_text = f"""
## Notebook 14B true-weight robustness methods

We performed a true weight-based transaction-cost and rebalance-frequency sensitivity analysis using the source-aware strict-test period from Notebook 13B. Strategy weights were loaded from the AURORA Notebook 10 and ROMA R2 output directories when available. The Notebook 14 issue of duplicate date-strategy labels was addressed by filtering strict-test rows where split indicators were present and then deduplicating by date and strategy name before reindexing.

For each strategy with available weights, we simulated daily portfolio returns from ETF return panels by holding target weights between rebalance dates. Rebalance frequencies were daily, weekly, monthly, and quarterly. Transaction costs were set to 0, 10, 25, and 50 basis points and were applied as turnover multiplied by the transaction-cost rate. Cash return was assumed to be zero.

Available AURORA true weights: `{has_aurora_weights}`. Available ROMA-P4 true weights: `{has_roma_weights}`. Available ROMA-B12 true weights: `{has_b12_weights}`. If AURORA weights are unavailable, AURORA-versus-ROMA true-weight sensitivity cannot be claimed from Notebook 14B and should remain based on the original Notebook 13B statistical comparison.
""".strip()

results_text = f"""
## Notebook 14B true-weight robustness results

The true-weight robustness analysis produced performance rows for {len(strategies_with_weights)} strategies across transaction-cost and rebalance-frequency scenarios. The available true-weight strategies were: {", ".join(strategies_with_weights) if strategies_with_weights else "none"}.

For the AURORA-versus-ROMA true-weight claim, AURORA higher Sharpe than ROMA-P4 held in {share_line("aurora_sharpe_gt_roma")}; AURORA higher Sortino held in {share_line("aurora_sortino_gt_roma")}; and AURORA less severe maximum drawdown held in {share_line("aurora_mdd_less_severe_than_roma")}. These indicators are only meaningful if AURORA true weights were available.

For the ROMA-versus-B12 true-weight claim, ROMA-P4 higher Sharpe than ROMA-B12 held in {share_line("roma_sharpe_gt_b12")}; ROMA-P4 less severe maximum drawdown than ROMA-B12 held in {share_line("roma_mdd_less_severe_than_b12")}; and ROMA-P4 higher total return than ROMA-B12 held in {share_line("roma_total_return_gt_b12")}. These indicators test whether the regime-template ROMA baseline remains weaker than the timing benchmark under alternative implementation assumptions.
""".strip()

limitations_text = """
## Notebook 14B robustness limitations

This notebook provides true weight-based sensitivity only for strategies whose daily weights are available and successfully standardized. If AURORA weights are not found in the stored Notebook 10 outputs, AURORA cannot be re-simulated under alternative transaction-cost and rebalance-frequency assumptions from this notebook alone. In that case, Notebook 14B should be used primarily to validate ROMA and benchmark implementation robustness and to document the missing AURORA weight limitation. A full AURORA true-weight sensitivity analysis requires exporting AURORA daily weights from Notebook 10.
""".strip()

recommended_paper_wording = """
## Recommended paper wording

As an additional robustness diagnostic, we re-simulated strategies with available daily weights under alternative transaction-cost and rebalance-frequency assumptions. The analysis deduplicated strategy weights by date and strategy name before reindexing, addressing the duplicate-label issue observed in the earlier sensitivity notebook. Results from this notebook are interpreted only for strategies with available true weights. When AURORA daily weights are unavailable, the AURORA-versus-ROMA robustness claim should not be restated as a true-weight sensitivity result; instead, the primary AURORA-versus-ROMA evidence remains the source-aware Notebook 13B paired bootstrap comparison.
""".strip()

write_markdown(MANUSCRIPT_RUN_DIR / "notebook14B_methods_text.md", methods_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14B_results_text.md", results_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14B_limitations_text.md", limitations_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook14B_recommended_paper_wording.md", recommended_paper_wording)

write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14B_methods_text.md", methods_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14B_results_text.md", results_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14B_limitations_text.md", limitations_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook14B_recommended_paper_wording.md", recommended_paper_wording)

# ============================================================
# 15. Output index
# ============================================================

print("\n" + "=" * 96)
print("Step 12: Creating output index")
print("=" * 96)

output_rows = [
    {
        "artifact_type": "table",
        "name": "original_notebook13B_performance",
        "path": str(TABLE_RUN_DIR / "notebook14B_00_original_notebook13B_performance.csv"),
        "description": "Original Notebook 13B performance for comparison.",
    },
    {
        "artifact_type": "table",
        "name": "aurora_weight_file_candidates",
        "path": str(TABLE_RUN_DIR / "notebook14B_01_aurora_weight_file_candidates.csv"),
        "description": "Auto-discovered AURORA weight-file candidates.",
    },
    {
        "artifact_type": "table",
        "name": "roma_weight_file_candidates",
        "path": str(TABLE_RUN_DIR / "notebook14B_02_roma_weight_file_candidates.csv"),
        "description": "Auto-discovered ROMA weight-file candidates.",
    },
    {
        "artifact_type": "table",
        "name": "weight_standardization_report",
        "path": str(TABLE_RUN_DIR / "notebook14B_03_weight_standardization_report.csv"),
        "description": "Report on weight standardization, strict-test filtering, and deduplication.",
    },
    {
        "artifact_type": "table",
        "name": "all_weights_standardized_deduplicated_strict",
        "path": str(TABLE_RUN_DIR / "notebook14B_04_all_weights_standardized_deduplicated_strict.csv"),
        "description": "Standardized and deduplicated strict-test weights.",
    },
    {
        "artifact_type": "table",
        "name": "weight_coverage_audit",
        "path": str(TABLE_RUN_DIR / "notebook14B_05_weight_coverage_audit.csv"),
        "description": "Audit of which strategies have true weights available.",
    },
    {
        "artifact_type": "table",
        "name": "true_weight_sensitivity_performance",
        "path": str(TABLE_RUN_DIR / "notebook14B_06_true_weight_sensitivity_performance.csv"),
        "description": "True weight-based sensitivity performance table.",
    },
    {
        "artifact_type": "table",
        "name": "true_weight_simulation_errors",
        "path": str(TABLE_RUN_DIR / "notebook14B_07_true_weight_simulation_errors.csv"),
        "description": "Simulation errors, if any.",
    },
    {
        "artifact_type": "table",
        "name": "missing_weight_strategy_diagnostic",
        "path": str(TABLE_RUN_DIR / "notebook14B_08_missing_weight_strategy_diagnostic.csv"),
        "description": "Strategies available in 13B but missing true weights.",
    },
    {
        "artifact_type": "table",
        "name": "true_weight_pairwise_robustness",
        "path": str(TABLE_RUN_DIR / "notebook14B_10_true_weight_pairwise_robustness.csv"),
        "description": "Pairwise true-weight robustness comparisons.",
    },
    {
        "artifact_type": "table",
        "name": "claim_stability_summary",
        "path": str(TABLE_RUN_DIR / "notebook14B_12_claim_stability_summary.csv"),
        "description": "Claim stability summary for available true-weight scenarios.",
    },
    {
        "artifact_type": "table",
        "name": "paper_true_weight_robustness_performance",
        "path": str(TABLE_RUN_DIR / "notebook14B_13_paper_true_weight_robustness_performance.csv"),
        "description": "Paper-ready true-weight robustness performance table.",
    },
    {
        "artifact_type": "table",
        "name": "paper_true_weight_robustness_pairwise",
        "path": str(TABLE_RUN_DIR / "notebook14B_14_paper_true_weight_robustness_pairwise.csv"),
        "description": "Paper-ready true-weight robustness pairwise table.",
    },
    {
        "artifact_type": "table",
        "name": "figure_index",
        "path": str(TABLE_RUN_DIR / "notebook14B_15_figure_index.csv"),
        "description": "Index of generated robustness figures.",
    },
    {
        "artifact_type": "manuscript",
        "name": "notebook14B_methods_text",
        "path": str(MANUSCRIPT_RUN_DIR / "notebook14B_methods_text.md"),
        "description": "Manuscript methods text for Notebook 14B.",
    },
    {
        "artifact_type": "manuscript",
        "name": "notebook14B_results_text",
        "path": str(MANUSCRIPT_RUN_DIR / "notebook14B_results_text.md"),
        "description": "Manuscript results text for Notebook 14B.",
    },
]

for _, r in figure_index.iterrows():
    output_rows.append({
        "artifact_type": "figure",
        "name": r["figure_id"],
        "path": r["path"],
        "description": r["caption"],
    })

output_index = pd.DataFrame(output_rows)

save_table(
    output_index,
    "notebook14B_output_index.csv",
    f"table_14B_16_output_index_{RUN_ID}.csv",
)

print("Output index:")
print(output_index.to_string(index=False))

# ============================================================
# 16. Validation report and manifest
# ============================================================

print("\n" + "=" * 96)
print("Step 13: Saving validation report and manifest")
print("=" * 96)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "14B_AURORA_ROMA_true_weight_sensitivity.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "True weight-based transaction-cost and rebalance-frequency sensitivity analysis. "
        "Fixes duplicate date-strategy labels from Notebook 14 and audits availability of AURORA weights."
    ),
    "input_paths": {
        "notebook13B_matrix": str(NOTEBOOK13B_MATRIX_PATH),
        "etf_return_panel": str(ETF_RETURN_PANEL_PATH),
        "aurora_n10_root": str(AURORA_N10_ROOT),
        "roma_r2_root": str(ROMA_R2_ROOT),
        "selected_aurora_weight_path": str(aurora_weight_path) if aurora_weight_path else None,
        "selected_roma_weight_path": str(roma_weight_path) if roma_weight_path else None,
    },
    "weight_availability": {
        "has_aurora_primary_weights": bool(has_aurora_weights),
        "has_roma_p4_weights": bool(has_roma_weights),
        "has_roma_b12_weights": bool(has_b12_weights),
        "strategies_with_weights": strategies_with_weights,
        "missing_weight_strategies": missing_weight_strategies,
    },
    "robustness_grid": {
        "transaction_cost_bps": TRANSACTION_COST_BPS_GRID,
        "rebalance_frequencies": list(REBALANCE_FREQUENCIES.keys()),
    },
    "claim_stability_summary": claim_summary.to_dict(orient="records"),
    "interpretation": (
        "Use true-weight results only for strategies with available daily weights. "
        "If AURORA weights are unavailable, do not claim AURORA true-weight robustness; "
        "primary AURORA evidence remains Notebook 13B paired bootstrap."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_RUN_DIR),
        "weights": str(WEIGHT_RUN_DIR),
        "figures": str(PAPER_FIGURE_DIR),
        "reports": str(REPORT_RUN_DIR),
        "manuscript_assets": str(MANUSCRIPT_RUN_DIR),
    },
    "educational_note": (
        "This notebook is for reproducible financial machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK14B_validation_report.json"
validation_report_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK14B_validation_report_{RUN_ID}.json"

write_json(validation_report_path, validation_report)
write_json(validation_report_global_path, validation_report)

manifest = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_RUN_DIR / "NOTEBOOK14B_file_manifest_SHA256.csv"
manifest_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK14B_file_manifest_SHA256_{RUN_ID}.csv"

manifest.to_csv(manifest_path, index=False)
manifest.to_csv(manifest_global_path, index=False)

# ============================================================
# 17. Final summary
# ============================================================

print("\n" + "=" * 96)
print("NOTEBOOK 14B COMPLETE")
print("=" * 96)
print("Run ID                                    :", RUN_ID)
print("Run root                                  :", RUN_ROOT)
print("Selected AURORA weight path               :", aurora_weight_path)
print("Selected ROMA weight path                 :", roma_weight_path)
print("AURORA primary weights available          :", has_aurora_weights)
print("ROMA-P4 weights available                 :", has_roma_weights)
print("ROMA-B12 weights available                :", has_b12_weights)
print("Strategies with true weights              :", len(strategies_with_weights))
print("True weight performance table             :", TABLE_RUN_DIR / "notebook14B_06_true_weight_sensitivity_performance.csv")
print("True weight pairwise robustness           :", TABLE_RUN_DIR / "notebook14B_10_true_weight_pairwise_robustness.csv")
print("Claim stability summary                   :", TABLE_RUN_DIR / "notebook14B_12_claim_stability_summary.csv")
print("Paper true-weight robustness performance  :", TABLE_RUN_DIR / "notebook14B_13_paper_true_weight_robustness_performance.csv")
print("Paper true-weight robustness pairwise     :", TABLE_RUN_DIR / "notebook14B_14_paper_true_weight_robustness_pairwise.csv")
print("Figure index                              :", TABLE_RUN_DIR / "notebook14B_15_figure_index.csv")
print("Output index                              :", TABLE_RUN_DIR / "notebook14B_output_index.csv")
print("Validation report                         :", validation_report_path)
print("Manifest                                  :", manifest_path)
print("=" * 96)

print("\nClaim stability summary:")
print(claim_summary.to_string(index=False))

if has_aurora_weights:
    print(
        "\nPaper-safe conclusion: Notebook 14B provides true weight-based robustness "
        "for AURORA and ROMA under transaction-cost and rebalance-frequency scenarios."
    )
else:
    print(
        "\nPaper-safe conclusion: Notebook 14B fixes the ROMA duplicate-label issue and "
        "provides true weight-based ROMA/benchmark sensitivity. However, AURORA true "
        "weight-based robustness cannot be claimed unless AURORA daily weights are "
        "exported from Notebook 10 and rerun here."
    )

Mounted at /content/drive
Notebook 14B: True Weight Transaction-Cost and Rebalance Sensitivity
Timestamp UTC              : 2026-06-26T00:40:04Z
Run ID                     : 20260626_004004
Notebook 13B matrix        : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916/returns/notebook13B_source_aware_strict_test_return_matrix.parquet
AURORA N10 root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
ROMA R2 root               : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_TWETF/aligned_allocation_backtest/run_20260625_025314
ETF return panel           : /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
Run root                   : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/true_weight_transaction_cost_rebalance_sensitivity/run_20260626_004004

Step 1: Loading source-aware returns and 